In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BostonCrimesExploration")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [ ]:
crimes = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/crime.csv")
)

offense_codes = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/offense_codes.csv")
)

In [ ]:
crimes_rowcount = crimes.count()
full_duplicate_count = crimes.count() - crimes.distinct().count()
print(f"Количество строк в crimes: {crimes_rowcount}, полных дублей: {full_duplicate_count}")
print(crimes.columns)

offense_codes_rowcount = offense_codes.count()
print(f"Количество строк в offense_codes: {offense_codes_rowcount}")

crimes.printSchema()

offense_codes.printSchema()

In [ ]:
crimes.show(5)


In [ ]:
offense_codes.show(5)

In [ ]:
from pyspark.sql import functions as F

result = (crimes.groupBy("INCIDENT_NUMBER")
                .agg(F.count("*").alias("number_count"))
                .filter(F.col("number_count") > 1)
                .orderBy(F.col("number_count").desc())
        )
incident_id = result.select("INCIDENT_NUMBER").first()["INCIDENT_NUMBER"]

crimes.filter(F.col("INCIDENT_NUMBER") == incident_id).show(truncate=False)


In [ ]:
from pyspark.sql import functions as F

offense_codes_rowcount = offense_codes.count()
unique_offense_codes = offense_codes.select("CODE").distinct().count()
print(f"Количество строк в offense_codes: {offense_codes_rowcount}, из них уникальных: {unique_offense_codes}")


result = (offense_codes.groupBy("CODE")
                .agg(F.count("*").alias("code_count"))
                .filter(F.col("code_count") > 1)
                .orderBy(F.col("code_count").desc())
                .limit(1)
        )

top_code = result.select("CODE").first()["CODE"]

result.show()

offense_codes.filter(F.col("CODE") == top_code).show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

crimes.select([F.count(F.when(F.isnull(F.col(c)),1)).alias(c) for c in crimes.columns]).show()


In [ ]:
print(crimes.count() - crimes.distinct().count())


In [ ]:
from pyspark.sql import functions as F

unique_offense_codes_names = (
      offense_codes
      .groupBy("CODE")
      .agg(
      F.countDistinct("NAME").alias("name_count"),
      F.collect_set("NAME").alias("names")
      )
      .filter(F.col("name_count") > 1)                 
)

unique_offense_codes_names.show(100)

In [ ]:
from pyspark.sql import functions as F

coordinates_stats = (
      crimes
      .agg(
      F.min("Lat").alias("min_lat"),
      F.max("Lat").alias("max_lat"),
      F.min("Long").alias("min_long"),
      F.max("Long").alias("max_long"),
      F.count(F.when(F.col("Lat") == 0, 1)).alias("zero_lat"),
      F.count(F.when(F.col("Long") == 0, 1)).alias("zero_long"),
      F.count(F.when(F.col("Lat") == -1, 1)).alias("minus1_lat"),
      F.count(F.when(F.col("Long") == -1, 1)).alias("minus1_long"),
      F.count(F.when((F.col("Long") == -1) & (F.col("Lat") == -1), 1)).alias("minus1_long_lat"),
      )                
)

coordinates_stats.show()

In [ ]:
from pyspark.sql import functions as F

crimes_clean = (
    crimes
    .dropDuplicates()
    .filter((F.col("DISTRICT") != "") & (F.col("DISTRICT").isNotNull()))
    .select(
        F.col("INCIDENT_NUMBER"),
        F.col("OFFENSE_CODE"),
        F.col("DISTRICT"),
        F.col("OCCURRED_ON_DATE"),
        F.col("YEAR"),
        F.col("MONTH"),
        F.when(F.col("Lat") == -1, None).otherwise(F.col("Lat")).alias("lat"),
        F.when(F.col("Long") == -1, None).otherwise(F.col("Long")).alias("lng")
    )    
)

print(crimes_clean.count())
print(crimes.count())

crimes_clean.select([F.count(F.when(F.isnull(F.col("DISTRICT")),1)).alias("DISTRICT NULL")]).show()


crimes_clean.printSchema()

In [ ]:
from pyspark.sql import functions as F

offense_codes_clean = (
    offense_codes
    .groupBy("CODE")
    .agg(F.min("NAME").alias("NAME"))
    .select(
        F.col("CODE"),
        F.col("NAME"),
        F.trim(F.split(F.col("NAME"), "-")[0]).alias("crime_type")
    )    
)

codes_duplicates = (
    offense_codes_clean
    .groupBy("CODE")
    .count()
    .filter(F.col("count") > 1)
)

codes_duplicates.show()

print(offense_codes.count())
print(offense_codes_clean.count())


offense_codes_clean.show(10, truncate=False)

In [ ]:
crimes_clean.select(
    F.count(F.when(F.col("DISTRICT").isNull(), 1)).alias("null_district"),
    F.count(F.when(F.trim(F.col("DISTRICT")) == "", 1)).alias("blank_district"),
    F.count(F.when(F.col("lat") == -1, 1)).alias("bad_lat"),
    F.count(F.when(F.col("lng") == -1, 1)).alias("bad_lng"),
).show()